In [38]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, HTML

BASE_DIR = Path(".").resolve()
MAPPINGS_DIR = BASE_DIR / "mappings_cox"


In [39]:

rows = []

for fp in sorted(MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 97 firm-variable mappings across 75 firms.


firm_id,variable,notes,final_choice,num_candidates
AGATE.CO,XSGA_COMPONENTS,"Candidate list missed the important overhead row 'Staff costs', which is a core SG&A/personnel expense. Chose it from the full A-column labels together with candidate 'Other external expenses' to capture operating overhead. Did not use subtotal 'Reported Operating Expense' and did not include depreciation/amortization rows. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Staff costs",Income Statement::Staff costs; Income Statement::Other external expenses,5
AGFEb.CO,COGS,"Candidate list for COGS is empty. 'External Costs' is the best available Income Statement label for direct operating/service costs and is more consistent with COGS than personnel or overhead rows, but classification is not fully certain and should be reviewed manually.",Income Statement::External Costs,0
ALKb.CO,XSGA_COMPONENTS,Selected broad operating overhead buckets rather than duplicated subcomponent labels. Added 'Research and development expenses' because R&D is separately reported and not obviously embedded in the selected overhead buckets; this aligns with the PROF rule using (XSGA_COMPONENTS - XRD). Excluded depreciation/amortization rows and totals. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Research and development expenses,Income Statement::Sales and marketing expenses; Income Statement::Administrative expenses; Income Statement::Research and development expenses,8
ALMB.CO,COGS,"No candidate was provided. For an insurance company, 'Insurance service expenses' is the closest analogue to direct cost of sales/cost of services. Chosen from the full A-column labels outside the candidate list, so manual review is required.",Income Statement::Insurance service expenses,0
ALSYDB.CO,BE,"Candidate list clearly misses the best BE label. Selected the balance-sheet parent equity line 'Shareholders of Sydbank A/S', which excludes minority interests and is preferable to 'Total Equity' to avoid double counting with MIB. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Shareholders of Sydbank A/S",Balance Sheet::Shareholders of Sydbank A/S,5
ALSYDB.CO,MIB,Candidate list misses the correct balance-sheet non-controlling interest line. Selected 'Minority shareholders' from the Balance Sheet as the appropriate MIB label. | Escape-hatch used (final_choice outside candidate list): [Balance Sheet] Minority shareholders,Balance Sheet::Minority shareholders,1
AMBUb.CO,XINT,"No clear single populated total interest-expense line is shown in the preview. 'Interest Expense' is the ideal label but appears empty here; 'Financial expenses' is the closest fallback single line, though it likely includes non-interest items. Manual review recommended to confirm the best interest measure.",Income Statement::Interest Expense; Income Statement::Financial expenses,6
AMBUb.CO,XSGA_COMPONENTS,No single explicit SG&A row exists. Selected the two main overhead buckets. Added Development costs as well because R&D is reported separately and is not obviously embedded in the selected selling/admin buckets; this allows PROF to compute SG&A excluding R&D via (XSGA_COMPONENTS - XRD). Excluded depreciation/amortization sublines and totals. | Escape-hatch used (final_choice outside candidate list): [Income Statement] Development costs,Income Statement::Selling and distribution costs; Income Statement::Management and administrative costs; Income Statement::Development costs,9
BAVA.CO,XSGA_COMPONENTS,"Candidate list appears to miss the important SG&A component row 'Sales and distribution costs'. Per SG&A rule, the explicit but mostly empty 'Selling, General & Admin' row was included together with populated overhead buckets for coverage. Included 'Research and Development Costs' as well because R&D is separately reported and not obviously embedded in the selected SG&A buckets, so XSGA - XRD can correctly remove it. | Escape-hatch used (fi

In [40]:
df_review = pd.DataFrame(rows)

df_xsga = (
    df_review[df_review["variable"] == "XSGA_COMPONENTS"]
    .sort_values(["firm_id"])
    .reset_index(drop=True)
)

print(f"XSGA_COMPONENTS manual reviews: {len(df_xsga)} rows across {df_xsga['firm_id'].nunique() if len(df_xsga) else 0} firms.")
display(df_xsga)

XSGA_COMPONENTS manual reviews: 58 rows across 58 firms.


,firm_id,variable,notes,final_choice,num_candidates
0,AGATE.CO,XSGA_COMPONENTS,Candidate list missed the important overhead r...,Income Statement::Staff costs; Income Statemen...,5
1,ALKb.CO,XSGA_COMPONENTS,Selected broad operating overhead buckets rath...,Income Statement::Sales and marketing expenses...,8
2,AMBUb.CO,XSGA_COMPONENTS,No single explicit SG&A row exists. Selected t...,Income Statement::Selling and distribution cos...,9
3,BAVA.CO,XSGA_COMPONENTS,Candidate list appears to miss the important S...,"Income Statement::Selling, General & Admin; In...",7
4,BETCO.CO,XSGA_COMPONENTS,Candidate list clearly misses an important ove...,Income Statement::Staff Costs; Income Statemen...,1
5,BIF.CO,XSGA_COMPONENTS,Candidate list appears to miss a major overhea...,Income Statement::External Expenses; Income St...,4
6,BIOPOR.CO,XSGA_COMPONENTS,No explicit consolidated SG&A row exists. Sele...,Income Statement::Sales and marketing costs; I...,6
7,BO.CO,XSGA_COMPONENTS,"Included 'Selling, General & Admin' because an...","Income Statement::Selling, General & Admin; In...",9
8,CARLb.CO,XSGA_COMPONENTS,Candidate list clearly misses the important SG...,Income Statement::Sales and distribution expen...,8
9,CBRAIN.CO,XSGA_COMPONENTS,Candidate list appears to miss the more comple...,Income Statement::Staff costs; Income Statemen...,4
